# BACEN ETL — API SGS + PySpark

Pipeline completo inspirado no ETL do IBGE: extração, camada raw, transformação, CSV e publicação opcional no PostgreSQL.

## 1. Importações e parâmetros

Com `ATUALIZAR_DADOS = False`, o pipeline reutiliza os JSONs mais recentes. Se eles ainda não existirem, a primeira extração acontece automaticamente. Use `True` para buscar dados novos.

In [5]:
from datetime import date
from etl_bacen import SERIES, executar_pipeline
import json
from pathlib import Path

ATUALIZAR_DADOS = False
PUBLICAR_POSTGRES = False
DATA_INICIAL = date(2016, 1, 1)
DATA_FINAL = date.today()
SERIES_ATIVAS = list(SERIES)

SERIES

{'selic_meta': {'codigo': 432,
  'nome': 'Meta Selic',
  'unidade': '% a.a.',
  'periodicidade': 'diaria'},
 'selic_efetiva': {'codigo': 1178,
  'nome': 'Taxa Selic efetiva anualizada',
  'unidade': '% a.a.',
  'periodicidade': 'diaria'},
 'cdi': {'codigo': 12,
  'nome': 'Taxa de juros CDI',
  'unidade': '% a.d.',
  'periodicidade': 'diaria'},
 'dolar_venda': {'codigo': 1,
  'nome': 'Taxa de câmbio - dólar americano (venda)',
  'unidade': 'R$/US$',
  'periodicidade': 'diaria'}}

## 2. Executar o pipeline

A função divide períodos longos, aplica retentativas, salva cada resposta em `data/raw`, transforma com PySpark, valida a chave e grava o CSV.

In [2]:
arquivo_processado = executar_pipeline(
    atualizar_dados=ATUALIZAR_DADOS,
    publicar_no_postgres=PUBLICAR_POSTGRES,
    data_inicial=DATA_INICIAL,
    data_final=DATA_FINAL,
    series_ativas=SERIES_ATIVAS,
)
arquivo_processado

selic_meta: reutilizando C:\Users\Gabriel\Desktop\Python_Treinamento\projeto-indicadores-publicos\APIs\BACEN\data\raw\bacen_selic_meta_432_20260805_140334.json
selic_efetiva: reutilizando C:\Users\Gabriel\Desktop\Python_Treinamento\projeto-indicadores-publicos\APIs\BACEN\data\raw\bacen_selic_efetiva_1178_20260805_140348.json
cdi: reutilizando C:\Users\Gabriel\Desktop\Python_Treinamento\projeto-indicadores-publicos\APIs\BACEN\data\raw\bacen_cdi_12_20260805_140403.json
dolar_venda: reutilizando C:\Users\Gabriel\Desktop\Python_Treinamento\projeto-indicadores-publicos\APIs\BACEN\data\raw\bacen_dolar_venda_1_20260805_140416.json
bacen_serie_historica: 11,842 linhas -> C:\Users\Gabriel\Desktop\Python_Treinamento\projeto-indicadores-publicos\APIs\BACEN\data\processed\bacen_serie_historica.csv


WindowsPath('C:/Users/Gabriel/Desktop/Python_Treinamento/projeto-indicadores-publicos/APIs/BACEN/data/processed/bacen_serie_historica.csv')

## 3. Validação da saída

In [3]:
import pandas as pd

resultado = pd.read_csv(arquivo_processado, parse_dates=["data_referencia"])
display(resultado.groupby(["serie_id", "indicador"]).agg(
    primeira_data=("data_referencia", "min"),
    ultima_data=("data_referencia", "max"),
    linhas=("valor", "size"),
).reset_index())
assert not resultado.duplicated(["serie_id", "data_referencia"]).any()
assert resultado["valor"].notna().all()

,serie_id,indicador,primeira_data,ultima_data,linhas
0,1,Taxa de câmbio - dólar americano (venda),2016-01-04,2026-08-05,2658
1,12,Taxa de juros CDI,2016-01-04,2026-08-04,2657
2,432,Meta Selic,2016-01-01,2026-08-05,3870
3,1178,Taxa Selic efetiva anualizada,2016-01-04,2026-08-04,2657


## 4. Criando e Tratando so DF's

In [6]:
# O CSV fica em data/processed; a camada bruta fica em data/raw.
RAW_DIR = Path(arquivo_processado).parent.parent / "raw"
arquivos_json = sorted(RAW_DIR.glob("*.json"))

if not arquivos_json:
    raise FileNotFoundError(f"Nenhum JSON encontrado em {RAW_DIR}")

# Um DataFrame Pandas independente para cada arquivo JSON.
# A chave do dicionário é o nome completo do arquivo, sem a extensão.
dfs_raw = {}
resumo_raw = []

for caminho in arquivos_json:
    registros = json.loads(caminho.read_text(encoding="utf-8"))
    df_raw = pd.DataFrame(registros)
    dfs_raw[caminho.stem] = df_raw

    resumo_raw.append({
        "arquivo": caminho.name,
        "linhas": len(df_raw),
        "colunas": ", ".join(df_raw.columns),
        "duplicadas_codigo_data": (
            int(df_raw.duplicated(["codigo", "data"]).sum())
            if {"codigo", "data"}.issubset(df_raw.columns) else None
        ),
        "valores_nulos": int(df_raw.isna().sum().sum()),
    })

display(pd.DataFrame(resumo_raw))
print(f"{len(dfs_raw)} DataFrames criados em dfs_raw")

,arquivo,linhas,colunas,duplicadas_codigo_data,valores_nulos
0,bacen_cdi_12_20260805_140403.json,2657,"serie, codigo, data, valor",0,0
1,bacen_dolar_venda_1_20260805_140416.json,2658,"serie, codigo, data, valor",0,0
2,bacen_selic_efetiva_1178_20260805_140348.json,2657,"serie, codigo, data, valor",0,0
3,bacen_selic_meta_432_20260805_140334.json,3870,"serie, codigo, data, valor",0,0


4 DataFrames criados em dfs_raw


### Inspecionar cada DataFrame bruto

Esta célula mantém os valores exatamente como vieram da API e apresenta tipos, nulos e uma amostra de cada arquivo.

In [7]:
for nome_arquivo, df_raw in dfs_raw.items():
    print("=" * 100)
    print(nome_arquivo)
    print("Tipos:")
    display(df_raw.dtypes.rename("dtype").to_frame())
    print("Nulos por coluna:")
    display(df_raw.isna().sum().rename("nulos").to_frame())
    display(df_raw.head())

bacen_cdi_12_20260805_140403
Tipos:


,dtype
serie,object
codigo,int64
data,object
valor,object


Nulos por coluna:


,nulos
serie,0
codigo,0
data,0
valor,0


,serie,codigo,data,valor
0,cdi,12,04/01/2016,0.052496
1,cdi,12,05/01/2016,0.052496
2,cdi,12,06/01/2016,0.052496
3,cdi,12,07/01/2016,0.052496
4,cdi,12,08/01/2016,0.052496


...

bacen_selic_meta_432_20260805_140334
Tipos:


,dtype
serie,object
codigo,int64
data,object
valor,object


Nulos por coluna:


,nulos
serie,0
codigo,0
data,0
valor,0


,serie,codigo,data,valor
0,selic_meta,432,01/01/2016,14.25
1,selic_meta,432,02/01/2016,14.25
2,selic_meta,432,03/01/2016,14.25
3,selic_meta,432,04/01/2016,14.25
4,selic_meta,432,05/01/2016,14.25


### Cópias tratadas para análise

A camada `dfs_raw` não é modificada. Os tratamentos abaixo são aplicados em cópias guardadas em `dfs_tratados`, permitindo comparar o valor original com datas e números convertidos.

In [10]:
def preparar_df_serie(nome_serie):
    """Consolida os arquivos raw de uma série e aplica os tratamentos."""
    config = SERIES[nome_serie]
    partes = []

    # A ordenação dos nomes garante que, em caso de várias extrações,
    # a versão de timestamp mais recente seja mantida.
    for nome_arquivo, df_raw in sorted(dfs_raw.items()):
        if "serie" not in df_raw.columns:
            continue
        parte = df_raw.loc[df_raw["serie"] == nome_serie].copy()
        if not parte.empty:
            parte["arquivo_raw"] = nome_arquivo
            partes.append(parte)

    if not partes:
        raise ValueError(f"Não encontrei dados raw para {nome_serie}")

    df = pd.concat(partes, ignore_index=True)
    df["data_referencia"] = pd.to_datetime(
        df["data"], format="%d/%m/%Y", errors="coerce"
    )
    df["valor"] = pd.to_numeric(
        df["valor"].astype("string").str.replace(",", ".", regex=False),
        errors="coerce",
    )
    df = (
        df.dropna(subset=["data_referencia", "valor"])
        .sort_values(["arquivo_raw", "data_referencia"])
        .drop_duplicates(["codigo", "data_referencia"], keep="last")
    )
    df["ano"] = df["data_referencia"].dt.year.astype("int16")
    df["mes"] = df["data_referencia"].dt.month.astype("int8")
    df["indicador"] = config["nome"]
    df["unidade"] = config["unidade"]
    df["periodicidade"] = config["periodicidade"]
    df = df.rename(columns={"codigo": "serie_id"})

    return (
        df[[
            "serie_id", "serie", "indicador", "unidade",
            "periodicidade", "data_referencia", "ano", "mes", "valor",
        ]]
        .sort_values("data_referencia")
        .reset_index(drop=True)
    )


# Cada série fica disponível em sua própria variável.
df_selic_meta = preparar_df_serie("selic_meta")
df_selic_efetiva = preparar_df_serie("selic_efetiva")
df_cdi = preparar_df_serie("cdi")
df_dolar_venda = preparar_df_serie("dolar_venda")

dfs_processados = {
    "selic_meta": df_selic_meta,
    "selic_efetiva": df_selic_efetiva,
    "cdi": df_cdi,
    "dolar_venda": df_dolar_venda,
}

display(pd.DataFrame([
    {"dataframe": nome, "linhas": len(df),
     "inicio": df["data_referencia"].min(),
     "fim": df["data_referencia"].max()}
    for nome, df in dfs_processados.items()
]))

,dataframe,linhas,inicio,fim
0,selic_meta,3870,2016-01-01,2026-08-05
1,selic_efetiva,2657,2016-01-04,2026-08-04
2,cdi,2657,2016-01-04,2026-08-04
3,dolar_venda,2658,2016-01-04,2026-08-05


### Salvar cada DataFrame em um CSV separado

Os quatro arquivos são gravados em `APIs/BACEN/data/processed`, com codificação compatível com Power BI e Excel.

In [11]:
PROCESSED_DIR = Path(arquivo_processado).parent
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

arquivos_processados = {}
for nome, df in dfs_processados.items():
    destino = PROCESSED_DIR / f"{nome}.csv"
    df.to_csv(destino, index=False, encoding="utf-8-sig")
    arquivos_processados[nome] = destino
    print(f"{nome}: {len(df):,} linhas -> {destino}")

arquivos_processados

selic_meta: 3,870 linhas -> C:\Users\Gabriel\Desktop\Python_Treinamento\projeto-indicadores-publicos\APIs\BACEN\data\processed\selic_meta.csv
selic_efetiva: 2,657 linhas -> C:\Users\Gabriel\Desktop\Python_Treinamento\projeto-indicadores-publicos\APIs\BACEN\data\processed\selic_efetiva.csv
cdi: 2,657 linhas -> C:\Users\Gabriel\Desktop\Python_Treinamento\projeto-indicadores-publicos\APIs\BACEN\data\processed\cdi.csv
dolar_venda: 2,658 linhas -> C:\Users\Gabriel\Desktop\Python_Treinamento\projeto-indicadores-publicos\APIs\BACEN\data\processed\dolar_venda.csv


{'selic_meta': WindowsPath('C:/Users/Gabriel/Desktop/Python_Treinamento/projeto-indicadores-publicos/APIs/BACEN/data/processed/selic_meta.csv'),
 'selic_efetiva': WindowsPath('C:/Users/Gabriel/Desktop/Python_Treinamento/projeto-indicadores-publicos/APIs/BACEN/data/processed/selic_efetiva.csv'),
 'cdi': WindowsPath('C:/Users/Gabriel/Desktop/Python_Treinamento/projeto-indicadores-publicos/APIs/BACEN/data/processed/cdi.csv'),
 'dolar_venda': WindowsPath('C:/Users/Gabriel/Desktop/Python_Treinamento/projeto-indicadores-publicos/APIs/BACEN/data/processed/dolar_venda.csv')}